<a href="https://colab.research.google.com/github/rfandan/PyTorch_Regression_template/blob/main/Pytorch_template_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import copy
import wandb
from torch.amp import autocast, GradScaler

In [ ]:
# ----------------------------
# SEEDING
# ----------------------------
torch.manual_seed(42)
np.random.seed(42)

# ----------------------------
# SYNTHETIC DATASET
# ----------------------------
X = np.random.randn(1000, 10)
y = (
    2 * X[:, 0] -
    1.5 * X[:, 1] +
    0.5 * X[:, 2] +
    np.random.randn(1000) * 0.1
) # y.shape = [1000]

In [ ]:
# ----------------------------
# SPLIT
# ----------------------------
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# ----------------------------
# TO TENSORS (REGRESSION FIX)
# ----------------------------
X_train = torch.tensor(X_train, dtype=torch.float32) #float32 for regression
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1) # y.shape = [1000,1]
y_val   = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
y_test  = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

In [ ]:
# ----------------------------
# DATASET CLASS
# ----------------------------
class CustomDataset(Dataset):
    """
    Generic dataset for regression and Gaussian regression tasks.

    Expected:
    - X: [N, features]
    - y: [N, 1] (float regression targets)

    Why float?
    - regression requires continuous values
    - Gaussian regression needs float targets for NLL loss
    """

    def __init__(self, X, y):
        assert len(X) == len(y), "X and y must have same length"

        # Ensure correct dtype for neural networks
        self.X = X.float()
        self.y = y.float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Returns one sample: (features, target)
        return self.X[idx], self.y[idx]

# ----------------------------
# DATASETS
# ----------------------------
train_dataset = CustomDataset(X_train, y_train)
val_dataset   = CustomDataset(X_val, y_val)
test_dataset  = CustomDataset(X_test, y_test)

# ----------------------------
# DEVICE
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# DATALOADERS
# ----------------------------
generator = torch.Generator()
generator.manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
# ============================================================
# TRAINER CLASS (GENERAL MLP VERSION: CLASSIFICATION + REGRESSION)
# ============================================================

class Trainer:
    def __init__(
        self,
        model,
        optimizer,
        criterion,
        device,

        train_loader,
        val_loader=None,
        test_loader=None,

        scheduler=None,
        metrics=None,

        epochs=10,
        patience=5,
        grad_clip=1.0,

        use_amp=True,
        compile_model=False,

        use_wandb=False,
        project_name="trainer-exp",

        log_every_n_epochs=1
    ):
        # ============================================================
        # DEVICE + MODEL
        # ============================================================
        self.device = device
        self.model = model.to(device)

        if compile_model:
            self.model = torch.compile(self.model)

        # ============================================================
        # CORE COMPONENTS
        # ============================================================
        self.optimizer = optimizer
        self.criterion = criterion
        self.scheduler = scheduler

        # ============================================================
        # DATA
        # ============================================================
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader

        # ============================================================
        # TRAIN CONFIG
        # ============================================================
        self.epochs = epochs
        self.patience = patience
        self.grad_clip = grad_clip
        self.log_every_n_epochs = log_every_n_epochs

        # ============================================================
        # AMP (MIXED PRECISION)
        # ============================================================
        self.use_amp = use_amp and device.type == "cuda"
        self.scaler = GradScaler(enabled=self.use_amp)

        # ============================================================
        # METRICS
        # ============================================================
        # NOTE:
        # metrics must be compatible with task (regression OR classification)
        self.metrics = metrics
        self.train_metrics = metrics or {}
        self.val_metrics = copy.deepcopy(metrics) if metrics else {}

        # ============================================================
        # EARLY STOPPING
        # ============================================================
        self.best_val_loss = float("inf")
        self.early_stop_counter = 0

        # ============================================================
        # W&B
        # ============================================================
        self.use_wandb = use_wandb
        if self.use_wandb:
            wandb.init(
                project=project_name,
                config={
                    "epochs": epochs,
                    "patience": patience,
                    "grad_clip": grad_clip,
                    "use_amp": self.use_amp,
                    "compile_model": compile_model
                }
            )

    # ============================================================
    # TRAIN ONE EPOCH
    # ============================================================
    def _train_one_epoch(self):
        self.model.train()

        total_loss = 0.0

        for metric in self.train_metrics.values():
            metric.reset()

        for x, y in self.train_loader:
            x = x.to(self.device, non_blocking=True)
            y = y.to(self.device, non_blocking=True)

            self.optimizer.zero_grad(set_to_none=True)

            # ----------------------------
            # FORWARD PASS
            # ----------------------------
            with autocast(device_type=self.device.type, enabled=self.use_amp):
                outputs = self.model(x)
                loss = self.criterion(outputs, y)

            # ----------------------------
            # BACKWARD PASS
            # ----------------------------
            self.scaler.scale(loss).backward()

            # ----------------------------
            # GRADIENT CLIPPING
            # ----------------------------
            if self.grad_clip is not None:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    self.grad_clip
                )

            # ----------------------------
            # OPTIMIZER STEP
            # ----------------------------
            self.scaler.step(self.optimizer)
            self.scaler.update()

            # ----------------------------
            # METRICS UPDATE
            # ----------------------------
            for metric in self.train_metrics.values():
                metric.update(outputs, y)

            total_loss += loss.detach()

            # ----------------------------
            # SCHEDULER STEP
            # ----------------------------
            if self.scheduler:
                self.scheduler.step()

        train_loss = total_loss / len(self.train_loader)

        train_results = {
            k: v.compute().item()
            for k, v in self.train_metrics.items()
        }

        return train_loss, train_results

    # ============================================================
    # VALIDATION
    # ============================================================
    def _validate(self):
        if self.val_loader is None:
            return None, None

        self.model.eval()

        total_loss = 0.0

        for metric in self.val_metrics.values():
            metric.reset()

        with torch.inference_mode():
            for x, y in self.val_loader:
                x = x.to(self.device)
                y = y.to(self.device)

                with autocast(device_type=self.device.type, enabled=self.use_amp):
                    outputs = self.model(x)
                    loss = self.criterion(outputs, y)

                for metric in self.val_metrics.values():
                    metric.update(outputs, y)

                total_loss += loss.detach()

        val_loss = total_loss / len(self.val_loader)

        val_results = {
            k: v.compute().item()
            for k, v in self.val_metrics.items()
        }

        return val_loss, val_results

    # ============================================================
    # FIT LOOP
    # ============================================================
    def fit(self):
        self.best_val_loss = float("inf")
        self.early_stop_counter = 0

        for epoch in range(self.epochs):

            train_loss, train_metrics = self._train_one_epoch()
            val_loss, val_metrics = self._validate()

            current_lr = self.optimizer.param_groups[0]["lr"]

            # ============================================================
            # EARLY STOPPING
            # ============================================================
            if val_loss is not None:
                if val_loss < self.best_val_loss:
                    self.best_val_loss = val_loss
                    self.early_stop_counter = 0

                    torch.save({
                        "epoch": epoch,
                        "model_state_dict": self.model.state_dict(),
                        "optimizer_state_dict": self.optimizer.state_dict(),
                        "scheduler_state_dict": self.scheduler.state_dict() if self.scheduler else None,
                        "best_val_loss": self.best_val_loss,
                    }, "best_model.pt")

                else:
                    self.early_stop_counter += 1

                if self.early_stop_counter >= self.patience:
                    print("🛑 Early stopping triggered")
                    break

            # Save last model
            torch.save({
                "epoch": epoch,
                "model_state_dict": self.model.state_dict(),
                "optimizer_state_dict": self.optimizer.state_dict(),
                "scheduler_state_dict": self.scheduler.state_dict() if self.scheduler else None,
            }, "last_model.pt")

            # ============================================================
            # W&B LOGGING
            # ============================================================
            if self.use_wandb:
                log_dict = {
                    "epoch": epoch,
                    "train_loss": train_loss.item(),
                    "val_loss": val_loss.item() if val_loss is not None else 0,
                    "lr": current_lr,
                }

                for k, v in train_metrics.items():
                    log_dict[f"train/{k}"] = v

                for k, v in val_metrics.items():
                    log_dict[f"val/{k}"] = v

                wandb.log(log_dict)

            # ============================================================
            # CONSOLE LOGGING
            # ============================================================
            if (epoch + 1) % self.log_every_n_epochs == 0:
                print(
                    f"Epoch {epoch+1}/{self.epochs} | "
                    f"Train Loss: {train_loss:.4f} | "
                    f"Val Loss: {val_loss:.4f if val_loss is not None else 'N/A'} | "
                    f"LR: {current_lr:.6f}"
                )

        print("✅ Training finished")

    # ============================================================
    # TEST
    # ============================================================
    def test(self):
        if self.test_loader is None:
            return

        checkpoint = torch.load("best_model.pt", map_location=self.device)
        self.model.load_state_dict(checkpoint["model_state_dict"])

        self.model.eval()

        total_loss = 0.0

        for metric in self.metrics.values():
            metric.reset()

        with torch.inference_mode():
            for x, y in self.test_loader:
                x = x.to(self.device)
                y = y.to(self.device)

                outputs = self.model(x)
                loss = self.criterion(outputs, y)

                total_loss += loss.item()

                for metric in self.metrics.values():
                    metric.update(outputs, y)

        results = {k: m.compute().item() for k, m in self.metrics.items()}

        print(f"\nTest Loss: {total_loss / len(self.test_loader):.4f}")
        for k, v in results.items():
            print(f"{k}: {v:.4f}")

In [ ]:
from torchmetrics import MeanSquaredError, MeanAbsoluteError, R2Score

metrics = {
    "mse": MeanSquaredError(),
    "mae": MeanAbsoluteError(),
    "r2": R2Score()
}

In [ ]:
# ----------------------------
# MAIN EXECUTION
# ----------------------------
torch.manual_seed(42)
model = Regressor(
    in_layer=10,
    hidden_layers=[64, 32],
    out_layer=1,
    activation="gelu",
    norm="layer",
    use_residual=False
)
criterion = nn.SmoothL1Loss()  # Huber loss
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9,0.999), eps=1e-8, weight_decay=0.01)

EPOCHS = 50

# Create Scheduler OUTSIDE the Trainer
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,
    anneal_strategy="cos",
    div_factor=25,
    final_div_factor=1e4
)

In [ ]:
print(model)

In [ ]:
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,

    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,

    scheduler=scheduler,
    metrics=metrics,
    epochs=EPOCHS, #IMPORTANT: epochs in both trainer and schedular should be same
    patience=10,
    use_amp=True,
    use_wandb=False
)

In [ ]:
# ----------------------------
# TRAINING
# ----------------------------
torch.manual_seed(42)
trainer.fit()

In [ ]:
# ----------------------------
# TESTING
# ----------------------------
trainer.test()